# Solutions — Diagnostic 2 — Collections

Exercise notebook: `00_diagnostic/diagnostic_02_collections.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Question 1 — 🟢 Easy · Inverted index

**Working solution**

In [ ]:
from collections import defaultdict

def inverted_index(docs):
    index = defaultdict(set)
    for doc_id, text in docs.items():
        for word in text.lower().split():
            index[word].add(doc_id)
    return {w: sorted(ids) for w, ids in sorted(index.items())}

**Explanation**

`defaultdict(set)` collects and dedupes; one dict comprehension sorts keys and values.

**Complexity:** O(words).

**Common mistakes**

- Lists with duplicate ids.

## Question 2 — 🟡 Medium · Top-k frequent with deterministic ties

**Working solution**

In [ ]:
from collections import Counter

def top_k(items, k):
    return sorted(Counter(items).items(), key=lambda kv: (-kv[1], kv[0]))[:k]

**Explanation**

`Counter` + a composite sort key. `most_common` alone breaks ties by first appearance.

**Complexity:** O(n + u log u).

**Common mistakes**

- Relying on `most_common` for ties.

## Question 3 — 🟡 Medium · Merge overlapping intervals

**Working solution**

In [ ]:
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged

**Explanation**

Sort, then compare with the last merged interval only. Tuples sort by start then end.

**Complexity:** O(n log n).

**Common mistakes**

- Not handling a fully-contained interval (`max`).

## Question 4 — 🟠 Hard · Nested order summaries

**Working solution**

In [ ]:
from collections import defaultdict

def customer_summary(orders):
    stats = defaultdict(lambda: {"orders": 0, "items": 0, "spend": 0, "_by_item": defaultdict(float)})
    for o in orders:
        s = stats[o["customer"]]
        s["orders"] += 1
        for it in o["items"]:
            qty = it.get("qty", 1)
            s["items"] += qty
            if o["status"] == "PAID":
                s["spend"] += it["price"] * qty
                s["_by_item"][it["name"]] += it["price"] * qty
    result = {}
    for customer in sorted(stats):
        s = stats[customer]
        by_item = s.pop("_by_item")
        s["top_item"] = min(by_item, key=lambda n: (-by_item[n], n)) if by_item else None
        result[customer] = dict(s)
    return result

**Explanation**

Nested accumulators per key; 'top by value with alphabetical tiebreak' is `min` with key `(-value, name)`.

**Complexity:** O(items).

**Common mistakes**

- Counting refunded spend.
- Non-deterministic tie-breaks.